# 05 — Financial Analysis

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path("../.." ).resolve()))
sys.path.insert(0, str(Path(".").resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.evaluation import ResultsManager

RESULTS_DIR = Path("results")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

rm = ResultsManager(RESULTS_DIR)
df = rm.load_all_results()
print(f"Loaded {len(df)} experiments")

# Financial metric columns
fin_cols = [c for c in df.columns if c.startswith("fin_")]
print(f"Financial metrics: {fin_cols}")

## 1. Sharpe Ratio Distribution

In [ ]:
sharpe_col = "fin_sharpe"
if sharpe_col not in df.columns:
    print(f"'{sharpe_col}' not found. Run experiments first.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # By mode
    ax = axes[0]
    mode_order = ["ml", "raw", "db4", "learned_wavelet"]
    valid_modes = [m for m in mode_order if m in df["mode"].unique()]
    sns.boxplot(data=df, x="mode", y=sharpe_col, order=valid_modes,
                palette=["#2ecc71","#95a5a6","#3498db","#e74c3c"], ax=ax)
    ax.axhline(0, ls="--", color="k", lw=0.8, alpha=0.5)
    ax.set_title("Sharpe Ratio by Mode")
    ax.set_xlabel("")
    
    # By model
    ax = axes[1]
    sns.boxplot(data=df, x="model_name", y=sharpe_col, ax=ax,
                palette="Set2")
    ax.axhline(0, ls="--", color="k", lw=0.8, alpha=0.5)
    ax.set_title("Sharpe Ratio by Model")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=30)
    
    plt.suptitle("Sharpe Ratio Distribution", fontsize=12)
    plt.tight_layout()
    plt.show()
    
    print(f"\nMean Sharpe by mode:\n{df.groupby('mode')[sharpe_col].mean().round(3)}")

## 2. Full Financial Metrics Table

In [ ]:
fin_summary = (
    df.groupby(["model_name", "mode"])[fin_cols]
    .mean()
    .round(4)
)
print("Financial metrics by model × mode (mean across all tickers):")
print(fin_summary.to_string())

## 3. Sharpe vs F1-macro Scatter

In [ ]:
if "fin_sharpe" in df.columns and "ml_f1_macro" in df.columns:
    fig, ax = plt.subplots(figsize=(10, 6))
    modes = df["mode"].unique()
    palette = {"ml": "#2ecc71", "raw": "#95a5a6", "db4": "#3498db", "learned_wavelet": "#e74c3c"}
    
    for mode in modes:
        sub = df[df["mode"] == mode]
        ax.scatter(sub["ml_f1_macro"], sub["fin_sharpe"],
                   label=mode, alpha=0.6, s=40,
                   color=palette.get(mode, "gray"))
    
    ax.axhline(0, ls="--", color="k", lw=0.8, alpha=0.4)
    ax.set_xlabel("F1-macro")
    ax.set_ylabel("Sharpe Ratio")
    ax.set_title("F1-macro vs Sharpe Ratio — all experiments")
    ax.legend(title="mode")
    plt.tight_layout()
    plt.show()

## 4. Best Configurations by Financial Metric

In [ ]:
for metric in ["fin_sharpe", "fin_calmar", "fin_cagr", "fin_win_rate"]:
    if metric not in df.columns:
        continue
    best = df.nlargest(5, metric)[["ticker", "model_name", "mode", metric, "ml_f1_macro"]]
    print(f"\nTop 5 by {metric}:")
    print(best.round(4).to_string(index=False))

## 5. Risk Metrics — MDD and VaR

In [ ]:
risk_cols = [c for c in ["fin_max_drawdown", "fin_var_95", "fin_cvar_95"] if c in df.columns]

if risk_cols:
    fig, axes = plt.subplots(1, len(risk_cols), figsize=(5 * len(risk_cols), 5))
    if len(risk_cols) == 1:
        axes = [axes]
    mode_order = ["ml", "raw", "db4", "learned_wavelet"]
    valid_modes = [m for m in mode_order if m in df["mode"].unique()]
    palette = ["#2ecc71","#95a5a6","#3498db","#e74c3c"][:len(valid_modes)]
    
    for ax, col in zip(axes, risk_cols):
        sns.boxplot(data=df, x="mode", y=col, order=valid_modes, palette=palette, ax=ax)
        ax.set_title(col.replace("fin_", "").replace("_", " ").title())
        ax.set_xlabel("")
        ax.tick_params(axis="x", rotation=30)
    
    plt.suptitle("Risk Metrics by Mode", fontsize=12)
    plt.tight_layout()
    plt.show()

## 6. Per-Ticker Best Sharpe

In [ ]:
if "fin_sharpe" in df.columns:
    best_sharpe = (
        df.loc[df.groupby("ticker")["fin_sharpe"].idxmax()]
        [["ticker", "model_name", "mode", "fin_sharpe", "fin_max_drawdown", "fin_calmar", "ml_f1_macro"]]
        .set_index("ticker")
        .sort_values("fin_sharpe", ascending=False)
    )
    print("Best Sharpe per ticker:")
    print(best_sharpe.round(4).to_string())
    
    # Bar chart
    fig, ax = plt.subplots(figsize=(14, 5))
    best_sharpe["fin_sharpe"].plot(kind="bar", ax=ax,
                                   color=["#27ae60" if v > 0 else "#e74c3c" 
                                          for v in best_sharpe["fin_sharpe"]])
    ax.axhline(0, ls="--", color="k", lw=0.8)
    ax.set_title("Best Sharpe Ratio per Ticker (best model/mode)")
    ax.set_xlabel("Ticker")
    ax.set_ylabel("Sharpe Ratio")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

## 7. Profit Factor & Win Rate Analysis

In [ ]:
pf_col  = "fin_profit_factor"
wr_col  = "fin_win_rate"
if pf_col in df.columns and wr_col in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    mode_order = ["ml", "raw", "db4", "learned_wavelet"]
    valid_modes = [m for m in mode_order if m in df["mode"].unique()]
    palette = ["#2ecc71","#95a5a6","#3498db","#e74c3c"][:len(valid_modes)]
    
    sns.boxplot(data=df[df[pf_col] < 5], x="mode", y=pf_col,
                order=valid_modes, palette=palette, ax=axes[0])
    axes[0].set_title("Profit Factor by Mode (clipped at 5)")
    axes[0].axhline(1, ls="--", color="k", lw=0.8)
    axes[0].set_xlabel("")
    
    sns.boxplot(data=df, x="mode", y=wr_col,
                order=valid_modes, palette=palette, ax=axes[1])
    axes[1].set_title("Win Rate by Mode")
    axes[1].axhline(0.5, ls="--", color="k", lw=0.8)
    axes[1].set_xlabel("")
    
    plt.tight_layout()
    plt.show()